<a href="https://colab.research.google.com/github/PioBasile/ProjetMachineLearning/blob/main/ProjetMachineLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd
import re
import spacy
"""
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Téléchargement des ressources NLTK nécessaires
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
"""
# Avis de Reyder sur les algo de classification :
# K-NN : nul les tweet on des context different on ne peut pas les classer par similarité
# Naive Bayésiene : Bien, on a des classes précises et distincte
# Arbre décisionnel : Pas mal, mais il faudrait faire plein de sous classifications

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
path = '/content/drive/MyDrive/ColabNotebooks/scitweets_export.tsv'

# sep='\t' indique que le séparateur est une tabulation
df = pd.read_csv(path, sep='\t')

# 1 -- NETOYAGE TEXTE ------------

contractions_map = {
    ("don't", "dont"): "do not",
    ("doesn't", "doesnt"): "does not",
    ("can't", "cant"): "cannot",
    ("won't", "wont"): "will not",
    ("i'm", "im"): "i am",
    ("it's", "its"): "it is",
    ("you're", "youre"): "you are",
    ("they're", "theyre"): "they are",
    ("we're",): "we are",
    ("that's", "thats"): "that is",
    ("what's", "whats"): "what is"
}

def expand_contractions(text, cmap=contractions_map):
    flat_map = {k: v for keys, v in cmap.items() for k in keys}

    text = text.replace("’", "'")
    tokens = text.split()
    expanded = []

    for t in tokens:
        key = t.lower()
        if key in flat_map:
            expanded.extend(flat_map[key].split())
        else:
            expanded.append(t)

    return " ".join(expanded)

def get_words_list(df):
    words_list = []
    for text in df['text']:
        words_list.extend(text.split())
    return words_list


def vectoriser(df, wordlist):

    total = []

    for text in df:

      text = expand_contractions(text)
      words = text.split(' ')
      vector = []

      for word in wordlist:
        if word in words:
          vector.append(1)
        else:
          vector.append(0)

      total.append(vector)

    return total

words_list = get_words_list(df)
vectors = vectoriser(df['text'], words_list)



In [ ]:
"""
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def advanced_text_processing(text):
    if not isinstance(text, str):
        return ""

    # tokenisation
    words = text.lower().split()

    # élimination des stop_words
    cleaned_words =[]
    for word in words:
        if word not in stop_words:
            # lemmatisation
            lemma = lemmatizer.lemmatize(word)
            cleaned_words.append(lemma)

    return " ".join(cleaned_words)
"""
nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

def advanced_text_processing(text):
    """Lemmatisation et suppression des Stop Words avec spaCy"""
    if not text:
        return ""

    # On passe le texte en minuscules à spaCy
    doc = nlp(text.lower())

    cleaned_words =[]
    for token in doc:
        # On exclut les stop words, la ponctuation et les espaces vides
        if not token.is_stop and not token.is_punct and not token.is_space:
            # On garde le lemme (token.lemma_)
            cleaned_words.append(token.lemma_)

    return " ".join(cleaned_words)

df['text_fully_cleaned'] = df['text_cleaned'].apply(advanced_text_processing)

display(df[['text_cleaned', 'text_fully_cleaned']])

,text_cleaned,text_fully_cleaned
0,Knees are a bit sore. i guess that is a sign t...,knee bit sore guess sign recent treadmilling work
1,McDonald's breakfast stop then the gym,mcdonald breakfast stop gym
2,Can any Gynecologist with Cancer Experience ex...,gynecologist cancer experience explain danger ...
3,Couch-lock highs lead to sleeping in the couch...,couch lock high lead sleep couch get to stop shit
4,Does daily routine help prevent problems with ...,daily routine help prevent problem bipolar dis...
...,...,...
1135,i am sorry but we DO NOT have 14 of million de...,sorry 14 million dead covid total death u.s ac...
1136,Dear NIN applicants you can kindly download th...,dear nin applicant kindly download enrolment f...
1137,Whats the uber support team email address?,s uber support team email address
1138,House passes bill to increase stimulus checks ...,house pass bill increase stimulus check 2000


### Test Naive Bayes



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Il faut vectoriser pour que le model puisse lire la data et on utilise TF-IDF (proposé par gémini car très populaire)
# TF mesure la fréquence des mots et IDF mesure l'importance ( et apparait beaucoup mais ne donne pas d'indication)

vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8) #max features = top 5000 mots
X = vectorizer.fit_transform(df['text_fully_cleaned'])


In [ ]:
from sklearn.model_selection import train_test_split

target_columns = ['science_related', 'scientific_claim', 'scientific_reference', 'scientific_context']

# Vérifier si toutes les colonnes cibles existent dans le DataFrame
if not all(col in df.columns for col in target_columns):
    missing_cols = [col for col in target_columns if col not in df.columns]
    raise KeyError(f"Attention: Les colonnes cibles suivantes sont manquantes: {missing_cols}. Veuillez vous assurer qu'elles existent dans votre DataFrame 'df' pour la classification multi-label.")

y = df[target_columns]
print(f"Colonnes cibles utilisées pour la classification multi-label: {target_columns}")

# Division des données en ensembles d'entraînement (80%) et de test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"{X_train.shape}") #Shape (912, 467) = 912 tweet pour 467 mot classé par TF-IDF

Colonnes cibles utilisées pour la classification multi-label: ['science_related', 'scientific_claim', 'scientific_reference', 'scientific_context']
(912, 467)


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
from sklearn.multioutput import MultiOutputClassifier

#on envoie les données dans le model deja existant

naive_bayes = MultinomialNB()
classifier = MultiOutputClassifier(naive_bayes)
classifier.fit(X_train, y_train)

print("Modèle Naive Bayes multi-label entraîné avec succès.")

Modèle Naive Bayes multi-label entraîné avec succès.


In [ ]:

y_pred = classifier.predict(X_test)

accuracy_exact_match = accuracy_score(y_test, y_pred)
print(f"Précision du modèle (Exact Match): {accuracy_exact_match:.2f}")



Précision du modèle (Exact Match): 0.64
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [1. 0. 1. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [1. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [1. 0. 0. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [1. 0. 0. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [1. 1. 0. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [1. 1. 1. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [1. 0. 0. 0.]


In [ ]:
#@title Entrez votre tweet ici et exécutez la cellule !
custom_tweet = "The blood in my penis moves whenever i see Pio https://t.co/3312423" #@param {type:"string"}

if not custom_tweet:
    print("Veuillez entrer un texte dans le champ 'custom_tweet' pour effectuer une prédiction.")
else:
    cleaned_custom_tweet = clean_tweet_text(custom_tweet)
    fully_cleaned_custom_tweet = advanced_text_processing(cleaned_custom_tweet)

    print(f"Texte original: {custom_tweet}")
    print(f"Texte nettoyé: {cleaned_custom_tweet}")
    print(f"Texte lemmatisé: {fully_cleaned_custom_tweet}")


    if 'best_model' in globals():
        prediction = best_model.predict([fully_cleaned_custom_tweet])
        print("Utilisation du meilleur modèle tuné pour la prédiction.")
    elif 'classifier' in globals() and 'vectorizer' in globals():
        vectorized_custom_tweet = vectorizer.transform([fully_cleaned_custom_tweet])
        prediction = classifier.predict(vectorized_custom_tweet)
        print("Utilisation du modèle Naive Bayes de base pour la prédiction.")
    else:
        print("Erreur: Aucun modèle entraîné (ni 'best_model', ni 'classifier'/'vectorizer') n'est disponible.")
        prediction = None

    if prediction is not None:
        results = {}
        for i, col in enumerate(target_columns):
            results[col] = "Oui" if prediction[0, i] == 1 else "Non"

        print("\n--- Prédictions du Modèle ---")
        for label, result in results.items():
            print(f"{label}: {result}")

Texte original: The blood in my penis moves whenever i see Pio https://t.co/3312423
Texte nettoyé: The blood in my penis moves whenever i see Pio
Texte lemmatisé: blood penis move pio
Utilisation du modèle Naive Bayes de base pour la prédiction.

--- Prédictions du Modèle ---
science_related: Oui
scientific_claim: Oui
scientific_reference: Non
scientific_context: Non
